# Bedrock ModelBuilder Example


In [1]:
# Configure AWS credentials and region
! ada credentials update --provider=isengard --account=099324990371 --role=Admin --profile=default --once
! aws configure set region us-east-1

2026/03/16 14:33:32 Refreshing aws credentials for default
2026/03/16 14:33:33 Successfully refreshed aws credentials for default


In [2]:
# Setup
import boto3
import json
import time
import random
from sagemaker.core.resources import TrainingJob
from sagemaker.serve.bedrock_model_builder import BedrockModelBuilder

[03/16/26 14:33:37] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=778733;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=885530;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/twillit/Library/Application Support/sagemaker/config.yaml


In [3]:
# Configuration
TRAINING_JOB_NAME = 'my-lora-run-tpnld-1773683343850'
ROLE_ARN = "arn:aws:iam::099324990371:role/AmazonSageMaker-ExecutionRole-20260219T233135"
REGION = 'us-east-1'
BUCKET = 'sagemaker-us-east-1-099324990371'

In [4]:
# Step 1: Get training job and prepare model path
training_job = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
print(f"Training job status: {training_job.training_job_status}")

# Use the hf_merged directory which has complete HuggingFace format
base_s3_path = training_job.model_artifacts.s3_model_artifacts
hf_model_path = base_s3_path.rstrip('/') + '/checkpoints/hf_merged/'
print(f"Using HF model path: {hf_model_path}")

[03/16/26 14:33:39] WARNING  No region provided. Using default region.                                 ]8;id=432135;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=278513;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#356\356]8;;\

                    INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=134958;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=705239;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#370\370]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=665312;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=477402;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

Training job status: Completed
Using HF model path: s3://sagemaker-us-east-1-099324990371/model-customization/output-artifacts/my-lora-run-tpnld-1773683343850/output/model/checkpoints/hf_merged/


In [5]:
# Step 2: Verify required files exist
s3_client = boto3.client('s3', region_name=REGION)

required_files = ['config.json', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']
model_prefix = hf_model_path.replace(f's3://{BUCKET}/', '')

print("Checking required files:")
for file in required_files:
    try:
        s3_client.head_object(Bucket=BUCKET, Key=model_prefix + file)
        print(f"✅ {file}")
    except:
        print(f"❌ {file} - MISSING")

[03/16/26 14:33:40] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=44052;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=307088;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

Checking required files:
❌ config.json - MISSING
❌ tokenizer.json - MISSING
❌ tokenizer_config.json - MISSING
❌ model.safetensors - MISSING


In [6]:
# Step 3: Create missing tokenizer files if needed
def ensure_tokenizer_files():
    # Create added_tokens.json (usually empty for Llama)
    try:
        s3_client.head_object(Bucket=BUCKET, Key=model_prefix + 'added_tokens.json')
        print("✅ added_tokens.json exists")
    except:
        s3_client.put_object(
            Bucket=BUCKET,
            Key=model_prefix + 'added_tokens.json',
            Body=json.dumps({}),
            ContentType='application/json'
        )
        print("✅ Created added_tokens.json")

ensure_tokenizer_files()

✅ added_tokens.json exists


In [ ]:
# Debug: Check what's actually in the S3 bucket
print("Checking S3 structure...")
base_prefix = base_s3_path.replace(f's3://{BUCKET}/', '')
print(f"Base prefix: {base_prefix}")

# List files to see the actual structure
response = s3_client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=base_prefix,
    Delimiter='/'
)

print("Contents:")
if 'Contents' in response:
    for obj in response['Contents'][:10]:  # Show first 10 files
        print(f"  {obj['Key']}")

# Check specifically for hf_merged directory
hf_merged_prefix = base_prefix.rstrip('/') + '/checkpoints/hf_merged/'
print(f"\nChecking hf_merged path: {hf_merged_prefix}")

try:
    response = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=hf_merged_prefix)
    if 'Contents' in response:
        print("Files in hf_merged:")
        for obj in response['Contents']:
            file_name = obj['Key'].replace(hf_merged_prefix, '')
            print(f"  {file_name}")
            
        # Now copy with correct paths
        for obj in response['Contents']:
            source_key = obj['Key']
            file_name = source_key.replace(hf_merged_prefix, '')
            dest_key = base_prefix.rstrip('/') + '/' + file_name
            
            try:
                s3_client.copy_object(
                    Bucket=BUCKET,
                    CopySource={'Bucket': BUCKET, 'Key': source_key},
                    Key=dest_key
                )
                print(f"✅ Copied {file_name}")
            except Exception as e:
                print(f"❌ Failed to copy {file_name}: {e}")
    else:
        print("No files found in hf_merged directory")
except Exception as e:
    print(f"Error: {e}")

Checking S3 structure...
Base prefix: model-customization/output-artifacts/my-lora-run-tpnld-1773683343850/output/model
Contents:

Checking hf_merged path: model-customization/output-artifacts/my-lora-run-tpnld-1773683343850/output/model/checkpoints/hf_merged/
Files in hf_merged:
  added_tokens.json
✅ Copied added_tokens.json


In [8]:
# Step 4: Create Bedrock model builder and deploy
job_name = f"bedrock-nova-import-{random.randint(1000, 9999)}"
print(f"Job name: {job_name}")

# Create builder with correct model path
bedrock_builder = BedrockModelBuilder(
    model=training_job
)

# Deploy to Bedrock
deployment_result = bedrock_builder.deploy(
    job_name=job_name,
    imported_model_name=job_name,
    role_arn=ROLE_ARN,
    custom_model_name=job_name
)

model_arn = deployment_result['modelArn']
print(model_arn)

Job name: bedrock-nova-import-4982


[03/16/26 14:33:41] INFO     S3 artifacts path:                                        ]8;id=917035;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py\bedrock_model_builder.py]8;;\:]8;id=848667;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py#212\212]8;;\
                             s3://sagemaker-us-east-1-099324990371/model-customization                             
                             /output-artifacts/my-lora-run-tpnld-1773683343850/output/                             
                             model                                                                                 

                    INFO     Manifest path:                                            ]8;id=352000;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py\bedrock_model_builder.py]8;;\:]8;id=280360;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py#219\219]8;;\
                             s3://sagemaker-us-east-1-099324990371/model-customization                             
                             /output-artifacts/my-lora-run-tpnld-1773683343850/output/                             
                             output/manifest.json                                                                  

                    INFO     Looking for manifest at                                   ]8;id=681920;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py\bedrock_model_builder.py]8;;\:]8;id=157754;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py#226\226]8;;\
                             s3://sagemaker-us-east-1-099324990371/model-customization                             
                             /output-artifacts/my-lora-run-tpnld-1773683343850/output/                             
                             output/manifest.json                                                                  

[03/16/26 14:33:42] INFO     Manifest content: {'checkpoint_s3_bucket':                ]8;id=350064;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py\bedrock_model_builder.py]8;;\:]8;id=173941;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py#232\232]8;;\
                             's3://customer-escrow-099324990371-smtj-cc62fd20/my-lora-                             
                             run-tpnld-1773683343850/896'}                                                         

                    INFO     Checkpoint URI:                                           ]8;id=265208;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py\bedrock_model_builder.py]8;;\:]8;id=611184;file:///Users/twillit/.local/share/mise/installs/python/3.12.6/lib/python3.12/site-packages/sagemaker/serve/bedrock_model_builder.py#239\239]8;;\
                             s3://customer-escrow-099324990371-smtj-cc62fd20/my-lora-r                             
                             un-tpnld-1773683343850/896                                                            

arn:aws:bedrock:us-east-1:099324990371:custom-model/imported/fyjoelpl3jra


In [9]:
model_arn = deployment_result['modelArn']

In [10]:
# Create the custom model deployment
from uuid import uuid4
bedrock_client = boto3.client('bedrock', region_name=REGION)

# Wait for model to be Active before deploying
while True:
    status = bedrock_client.get_custom_model(modelIdentifier=model_arn).get("modelStatus")
    print(f"Model status: {status}")
    if status == "Active":
        break
    if status == "Failed":
        raise RuntimeError("Model creation failed")
    time.sleep(60)

# Now safe to create deployment
deploy_resp = bedrock_client.create_custom_model_deployment(
    modelDeploymentName=f"deployment-{job_name}",
    modelArn=model_arn,
    clientRequestToken=str(uuid4()),
)
deployment_arn = deploy_resp['customModelDeploymentArn']

Model status: Creating
Model status: Creating
Model status: Creating
Model status: Creating
Model status: Creating
Model status: Creating
Model status: Creating
Model status: Creating
Model status: Creating
Model status: Active


In [11]:
# Step 5: Wait for custom model creation to complete

print("Waiting for deployment to complete...")
while True:
    status = bedrock_client.get_custom_model_deployment(
        customModelDeploymentIdentifier=deployment_arn
    ).get("status")
    print(f"Deployment status: {status}")
    if status == "Active":
        break
    if status == "Failed":
        raise RuntimeError("Deployment failed")
    time.sleep(30)

Waiting for deployment to complete...
Deployment status: Creating
Deployment status: Creating
Deployment status: Creating
Deployment status: Creating
Deployment status: Creating
Deployment status: Creating
Deployment status: Active


In [12]:
# Step 6: Test inference with correct format
bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")
message = "What is the capital of France?"
print(f"Model Inference Message: {message}")
resp = bedrock_runtime.converse(
    modelId=deployment_arn,
    messages=[{"role": "user", "content": [{"text": message}]}],
    inferenceConfig={"maxTokens": 100, "temperature": 0.7},
)

response_str = resp["output"]["message"]["content"][0]["text"]
print(f"Model Response: {response_str}")

Model Inference Message: What is the capital of France?
Model Response: The capital of France is Paris. Paris is not only the political center of France but also a major cultural, historical, and economic hub. It is situated in the northern part of the country, along the Seine River. Paris is renowned for its iconic landmarks such as the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and the Champs-Élysées. The city is also famous for its influence on art, fashion, cuisine, and philosophy.
